In [15]:
from spectraformer.train import shard_batch
import jax
print("JAX devices: ", jax.devices())
num_devices = jax.device_count()
print("Number of devices: ", num_devices)
import jax.numpy as jnp
from spectraformer.input_pipeline import Batch, batch_sampler, preprocess_dataset
import xarray as xr

def create_test_batch():
    """Generate a batch divisible by device count"""
    num_devices = jax.device_count()
    batch_size = num_devices  # Or multiple of num_devices (e.g., 8 for 4 devices)
    return Batch(
        spectra=jnp.ones((batch_size, 812, 1)),      # Now size (4, 812, 1)
        masked_spectra=jnp.ones((batch_size, 812, 1)),
        wave_number=jnp.ones((812, 1)),
        mask=jnp.ones((812,))
    )

def test_sharding():
    actual_devices = len(jax.devices())
    test_batch = create_test_batch()
    
    try:
        sharded = shard_batch(test_batch, num_devices=actual_devices)
        print("Sharding successful!")
        print("Sharded batch shapes:")
        for k, v in sharded.items():
            print(f"{k}: {v.shape}")
    except Exception as e:
        print(f"Sharding failed: {str(e)}")

test_sharding()

JAX devices:  [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]
Number of devices:  4
Sharding successful!
Sharded batch shapes:
spectra: (4, 1, 812, 1)
masked_spectra: (4, 1, 812, 1)
wave_number: (4, 1, 812, 1)
mask: (4, 1, 812)


In [20]:
your_dataset = full_ds = preprocess_dataset(
        xr.load_dataarray("/home/dpoteryayev/SpectraFormer/data/SiC_19x10x3.nc")
    )

mask_windows = list(
        zip([1000, 2500], [1660, 2900])
    )

def test_real_data_sharding():
    # Load actual dataset
    real_batch = next(batch_sampler(your_dataset, mask_windows, batch_size=8))  # Multiple of 4
    sharded = shard_batch(real_batch, num_devices)
    print("Real data sharding successful!")

@jax.pmap
def dummy_forward(batch):
    return batch['spectra'].mean()  # Simple operation

test_real_data_sharding()


real_batch = next(batch_sampler(your_dataset, mask_windows, batch_size=8))  # Multiple of 4
sharded = shard_batch(real_batch, num_devices)
result = dummy_forward(sharded)
print("PMAP test result:", result)  # Should return array of 4 values

Dropped 0 spectra
Real data sharding successful!
PMAP test result: [0.20155333 0.21267682 0.20450467 0.20194742]
